# Phase 9-C Step 3: QLoRA FineTuning

**作成日**: 2026-03-01  
**プロジェクト**: experiments-local-llm  
**目的**: Qwen3-32B を QLoRA で FineTuning し、POI 回答品質を改善する

---

## 実験設計

| 項目 | 値 |
|------|----|
| ベースモデル | Qwen/Qwen3-32B (NF4 4bit) |
| 学習データ | C2 高品質回答 59件 (composite>=70, reasoning>=3) |
| 学習手法 | QLoRA (r=16, alpha=32) |
| GPU | A100 40GB (必須) |
| Train/Valid | 47/12 (80/20分割) |

## C3 目標値

| 指標 | C2 実績 | C3 目標 |
|------|---------|--------|
| composite_score | 70.4 | **75+** |
| reasoning_score | 3.07 | **3.5+** |
| evidence_score | 3.85 | **4.0+** |
| composite_success_rate | 83.1% | **88%+** |

## 0. リポジトリ同期

In [1]:
import os
import sys

# Colab環境の場合のみ実行
if 'google.colab' in sys.modules:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')

REPO_PATH = '/content/drive/MyDrive/experiments-local-llm'
BRANCH = 'feature/phase9c-step3-qlora'

print(f"Repository: {REPO_PATH}")
print(f"Target branch: {BRANCH}")
print()

# GitHub PAT認証
from getpass import getpass
github_token = getpass("GitHub Personal Access Token: ")
!git -C {REPO_PATH} remote set-url origin https://{github_token}@github.com/mopinfish/experiments-local-llm.git
print("GitHub HTTPS authentication configured")
print()

# Git設定
!git -C {REPO_PATH} config user.name "colab-runner"
!git -C {REPO_PATH} config user.email "colab@example.com"

# リモート最新取得
!git -C {REPO_PATH} stash --include-untracked -m "auto-stash before sync"
!git -C {REPO_PATH} fetch origin
!git -C {REPO_PATH} checkout {BRANCH}
!git -C {REPO_PATH} pull origin {BRANCH}
!git -C {REPO_PATH} stash pop 2>/dev/null || echo "No stash to pop"

print()
print("=" * 60)
!git -C {REPO_PATH} log --oneline -5
print()
!git -C {REPO_PATH} branch --show-current
print("=" * 60)
print("\n\u2705 リポジトリ同期完了")

Repository: /content/drive/MyDrive/experiments-local-llm
Target branch: feature/phase9c-step3-qlora

GitHub Personal Access Token: ··········
GitHub HTTPS authentication configured

Saved working directory and index state On phase9c-step3-qlora: auto-stash before sync
error: RPC failed; curl 16 Error in the HTTP2 framing layer
fatal: error reading section header 'acknowledgments'
Already on 'feature/phase9c-step3-qlora'
Your branch is up to date with 'origin/feature/phase9c-step3-qlora'.
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 842 bytes | 1024 bytes/s, done.
From https://github.com/mopinfish/experiments-local-llm
 * branch            feature/phase9c-step3-qlora -> FETCH_HEAD
   9963dfc..975386c  feature/phase9c-step3-qlora -> origin/feature/phase9c-step3-qlora
Updating 9963dfc..975386c
Fast-forward


## 1. 環境セットアップ

In [2]:
import sys
import os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    PROJECT_PATH = '/content/drive/MyDrive/experiments-local-llm'
    sys.path.insert(0, f'{PROJECT_PATH}/src')

    # パッケージインストール
    !pip install -q transformers accelerate bitsandbytes
    !pip install -q peft trl datasets
    !pip install -q torch
else:
    PROJECT_PATH = '..'
    sys.path.insert(0, f'{PROJECT_PATH}/src')

# GPU確認
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1024**3 if hasattr(torch.cuda.get_device_properties(0), 'total_mem') else torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")

    # A100必須チェック
    if 'A100' not in gpu_name:
        print(f"\n\u26a0\ufe0f WARNING: A100 GPU required for QLoRA training of Qwen3-32B!")
        print(f"  Current GPU: {gpu_name}")
        print(f"  Please switch to A100 runtime in Colab Pro.")
        raise RuntimeError(f"A100 GPU required, got {gpu_name}")
    else:
        print("\u2705 A100 GPU confirmed")

print(f"\nProject path: {PROJECT_PATH}")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 79.3 GB
✅ A100 GPU confirmed

Project path: /content/drive/MyDrive/experiments-local-llm


## 2. 学習データ準備

In [3]:
import json
import re

# --- 学習データ読み込み ---
training_data_file = f"{PROJECT_PATH}/data/phase9c_training_data.json"
print(f"Loading training data from: {training_data_file}")

with open(training_data_file, encoding="utf-8") as f:
    training_data = json.load(f)

train_samples = training_data["train"]
valid_samples = training_data["validation"]
metadata = training_data["metadata"]

print(f"\nDataset statistics:")
print(f"  Total selected: {metadata['selected_count']}")
print(f"  Train: {len(train_samples)}")
print(f"  Validation: {len(valid_samples)}")
print(f"  Selection criteria: composite>={metadata['selection_criteria']['composite_score_min']}, "
      f"reasoning>={metadata['selection_criteria']['reasoning_score_min']}")
print(f"  Level distribution: {metadata['level_distribution']}")

# サンプル表示
print(f"\nSample training data:")
for s in train_samples[:2]:
    print(f"  [{s['metadata']['test_id']}] {s['instruction'][:60]}...")
    print(f"    -> {s['output'][:80]}...")
    print()

Loading training data from: /content/drive/MyDrive/experiments-local-llm/data/phase9c_training_data.json

Dataset statistics:
  Total selected: 59
  Train: 47
  Validation: 12
  Selection criteria: composite>=70, reasoning>=3
  Level distribution: {'1': 9, '2': 9, '3': 18, '4': 16, '5': 7}

Sample training data:
  [MA-SBY-L1-03] 渋谷駅周辺のスターバックスはありますか？...
    -> 【結論】  
渋谷駅周辺にはスターバックスがあります。

【根拠】  
提供された検索結果に「スターバックス 飲食店/カフェ」という情報が5件表示されており、少...

  [MA-SBY-L2-02] 渋谷駅周辺のカフェとバー、どちらが多いですか？...
    -> 【結論】  
渋谷駅周辺ではカフェの方がバーよりも1件多く、カフェの数が多い。

【根拠】  
提供されたデータによると、渋谷駅周辺のカフェの総数は149件であ...



In [4]:
from datasets import Dataset

def format_for_training(sample):
    """Alpaca形式をチャット形式に変換"""
    # システムプロンプト（C1改善版と同一）
    system_prompt = """あなたは東京都内の主要駅周辺エリア（渋谷駅周辺、新宿駅周辺、池袋駅周辺、東京駅周辺）の地理情報に詳しいアシスタントです。
提供されたデータに基づいて、以下の構造で回答してください。

# 回答の構造
1. **結論**: 質問への直接的な回答を最初に述べる
2. **根拠**: データから得られた具体的な証拠を引用する
3. **補足**: 注意点や不確実な点があれば述べる

# 回答ルール
- 推論過程を明示する: 「したがって」「比較すると」「分析すると」「なぜなら」等の論理接続詞を使い、結論に至る過程を示す
- 根拠を具体的に引用する: POI名、座標(緯度, 経度)、距離(m)、件数を提供データから引用し、「データから」「検索結果に基づき」等で出典を明記する
- 数値は単位付きで示す: 距離はm、件数は件、座標は(35.xxx, 139.xxx)の形式で記載する
- 比較表現を使う: 「より多い」「最も近い」「〜倍」等の比較表現で差異を明確にする
- 不確実性を正直に示す: データで確認できない点は「ただし」「データの限界として」「可能性があります」「データからは確認できません」等で明記する
- 情報がない場合は「提供データからは確認できません」と正直に回答する"""

    return {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": sample["instruction"]},
            {"role": "assistant", "content": sample["output"]},
        ]
    }

# Dataset作成
train_formatted = [format_for_training(s) for s in train_samples]
valid_formatted = [format_for_training(s) for s in valid_samples]

train_dataset = Dataset.from_list(train_formatted)
valid_dataset = Dataset.from_list(valid_formatted)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Valid dataset: {len(valid_dataset)} samples")
print(f"\nSample messages structure:")
sample_msg = train_formatted[0]["messages"]
for msg in sample_msg:
    content_preview = msg['content'][:80] + '...' if len(msg['content']) > 80 else msg['content']
    print(f"  {msg['role']}: {content_preview}")

Train dataset: 47 samples
Valid dataset: 12 samples

Sample messages structure:
  system: あなたは東京都内の主要駅周辺エリア（渋谷駅周辺、新宿駅周辺、池袋駅周辺、東京駅周辺）の地理情報に詳しいアシスタントです。
提供されたデータに基づいて、以下の構造...
  user: 渋谷駅周辺のスターバックスはありますか？
  assistant: 【結論】  
渋谷駅周辺にはスターバックスがあります。

【根拠】  
提供された検索結果に「スターバックス 飲食店/カフェ」という情報が5件表示されており、少...


## 3. モデルロード (Qwen3-32B NF4)

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import gc
import warnings
warnings.filterwarnings('ignore')

# VRAMクリーンアップ（前回実行の残留メモリを確実に解放）
_cleanup_vars = ['model', 'tokenizer', 'trainer', 'ft_model', 'peft_model',
                 'base_model', 'inputs', 'outputs', 'train_result', 'eval_result']
for var_name in _cleanup_vars:
    if var_name in globals():
        try:
            obj = globals()[var_name]
            if hasattr(obj, 'cpu'):
                obj.cpu()
            del globals()[var_name]
        except Exception:
            pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.empty_cache()
    vram_used = torch.cuda.memory_allocated() / 1024**3
    print(f"VRAM after cleanup: {vram_used:.2f} GB")
    if vram_used > 5.0:
        print(f"\n🛑 ERROR: {vram_used:.1f} GB still in use after cleanup.")
        print("  This is leftover memory from a previous run that cannot be freed.")
        print()
        print("  ➡️  Please do: Runtime → Restart session")
        print("  ➡️  Then re-run all cells from the top (cell 0)")
        print()
        print("  Qwen3-32B NF4 requires ~20 GB clean VRAM to load.")
        raise RuntimeError(
            f"VRAM not clean ({vram_used:.1f} GB in use). "
            f"Restart runtime first: Runtime → Restart session"
        )

model_name = "Qwen/Qwen3-32B"
print(f"\nLoading {model_name}...")

# NF4量子化設定（A100向けにbfloat16を使用）
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
    padding_side="right"
)
print("Tokenizer loaded")

# pad_tokenの設定
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)
model.config.use_cache = False  # gradient checkpointing用
print("Model loaded (NF4 4-bit quantized)")

if torch.cuda.is_available():
    print(f"VRAM after model load: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

# Qwen3 thinking mode 無効化（monkey-patch）
original_apply = tokenizer.apply_chat_template
def patched_apply(*args, **kwargs):
    kwargs['enable_thinking'] = False
    return original_apply(*args, **kwargs)
tokenizer.apply_chat_template = patched_apply
print("✅ Qwen3 thinking mode disabled (monkey-patch)")

VRAM after cleanup: 0.00 GB

Loading Qwen/Qwen3-32B...
Tokenizer loaded


Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

Model loaded (NF4 4-bit quantized)
VRAM after model load: 17.92 GB
✅ Qwen3 thinking mode disabled (monkey-patch)


## 4. QLoRA設定

In [6]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# gradient checkpointing用の準備
model = prepare_model_for_kbit_training(model)

# LoRA設定
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

# 学習可能パラメータ数を表示
model.print_trainable_parameters()

if torch.cuda.is_available():
    print(f"VRAM after PEFT: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

trainable params: 134,217,728 || all params: 32,896,340,992 || trainable%: 0.4080
VRAM after PEFT: 21.34 GB


## 5. 学習実行

In [7]:
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

# チェックポイントディレクトリ
output_dir = f"{PROJECT_PATH}/phase9c_step3_checkpoints"
os.makedirs(output_dir, exist_ok=True)

# 学習パラメータ（59件の少量データに最適化）
sft_config = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,     # 実効バッチサイズ 4
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    bf16=True,                         # A100はbf16ネイティブサポート
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=0.3,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    logging_steps=5,
    max_length=1024,
    packing=False,
)

# 学習ステップ見積もり
steps_per_epoch = len(train_dataset) // (sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps)
total_steps = steps_per_epoch * sft_config.num_train_epochs
print(f"Training estimation:")
print(f"  Train samples: {len(train_dataset)}")
print(f"  Effective batch size: {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}")
print(f"  Steps per epoch: {steps_per_epoch}")
print(f"  Total epochs: {sft_config.num_train_epochs}")
print(f"  Total steps: ~{total_steps}")
print(f"  Estimated time: 30-60 min (A100)")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training estimation:
  Train samples: 47
  Effective batch size: 4
  Steps per epoch: 11
  Total epochs: 5
  Total steps: ~55
  Estimated time: 30-60 min (A100)


In [8]:
# SFTTrainer初期化・学習実行
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
)

print("Starting training...")
print("=" * 60)

train_result = trainer.train()

print("=" * 60)
print("Training complete!")
print(f"\nTraining metrics:")
for key, value in train_result.metrics.items():
    print(f"  {key}: {value}")

Tokenizing train dataset:   0%|          | 0/47 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/47 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting training...


Epoch,Training Loss,Validation Loss
1,0.815847,0.424537
2,0.331907,0.354829
3,0.196081,0.354599
4,0.122272,0.365711
5,0.086284,0.367147


Training complete!

Training metrics:
  train_runtime: 419.0233
  train_samples_per_second: 0.561
  train_steps_per_second: 0.143
  total_flos: 3.187529216695296e+16
  train_loss: 0.34722207188606263


In [9]:
# Validation loss確認（過学習チェック）
eval_result = trainer.evaluate()

print(f"\nValidation metrics:")
for key, value in eval_result.items():
    print(f"  {key}: {value}")

# 学習ログからloss推移を取得
log_history = trainer.state.log_history
train_losses = [(entry['step'], entry['loss']) for entry in log_history if 'loss' in entry]
eval_losses = [(entry['step'], entry['eval_loss']) for entry in log_history if 'eval_loss' in entry]

print(f"\nTraining loss history:")
for step, loss in train_losses:
    print(f"  Step {step}: {loss:.4f}")

print(f"\nEvaluation loss history:")
for step, loss in eval_losses:
    print(f"  Step {step}: {loss:.4f}")

# 過学習チェック
if eval_losses:
    best_eval_loss = min(loss for _, loss in eval_losses)
    last_eval_loss = eval_losses[-1][1]
    if last_eval_loss > best_eval_loss * 1.1:
        print(f"\n\u26a0\ufe0f WARNING: Possible overfitting detected!")
        print(f"  Best eval loss: {best_eval_loss:.4f}")
        print(f"  Last eval loss: {last_eval_loss:.4f}")
    else:
        print(f"\n\u2705 No significant overfitting (best={best_eval_loss:.4f}, last={last_eval_loss:.4f})")


Validation metrics:
  eval_loss: 0.3545990288257599
  eval_runtime: 5.4213
  eval_samples_per_second: 2.213
  eval_steps_per_second: 2.213

Training loss history:
  Step 5: 1.4112
  Step 10: 0.8158
  Step 15: 0.3827
  Step 20: 0.3319
  Step 25: 0.2754
  Step 30: 0.2013
  Step 35: 0.1961
  Step 40: 0.1553
  Step 45: 0.1223
  Step 50: 0.0960
  Step 55: 0.0925
  Step 60: 0.0863

Evaluation loss history:
  Step 12: 0.4245
  Step 24: 0.3548
  Step 36: 0.3546
  Step 48: 0.3657
  Step 60: 0.3671
  Step 60: 0.3546

✅ No significant overfitting (best=0.3546, last=0.3546)


## 6. アダプター保存

In [14]:
# Google Driveにアダプターを保存
adapter_save_path = "/content/drive/MyDrive/models/qwen3-32b-poi-qlora"
os.makedirs(adapter_save_path, exist_ok=True)

# best modelを保存（load_best_model_at_end=True なので既にベストモデル）
trainer.model.save_pretrained(adapter_save_path)
tokenizer.save_pretrained(adapter_save_path)

# 学習メタデータも保存
training_metadata = {
    "base_model": model_name,
    "quantization": "NF4 4-bit",
    "lora_r": lora_config.r,
    "lora_alpha": lora_config.lora_alpha,
    "target_modules": sorted(lora_config.target_modules),  # set→list変換
    "num_train_epochs": int(sft_config.num_train_epochs),
    "learning_rate": sft_config.learning_rate,
    "train_samples": len(train_dataset),
    "valid_samples": len(valid_dataset),
    "train_loss_final": train_losses[-1][1] if train_losses else None,
    "eval_loss_final": eval_losses[-1][1] if eval_losses else None,
    "eval_loss_best": best_eval_loss if eval_losses else None,
    "training_metrics": train_result.metrics,
    "loss_history": {
        "train": train_losses,
        "eval": eval_losses,
    },
}

with open(f"{adapter_save_path}/training_metadata.json", "w", encoding="utf-8") as f:
    json.dump(training_metadata, f, ensure_ascii=False, indent=2)

print(f"✅ Adapter saved to: {adapter_save_path}")
print(f"\nSaved files:")
for fname in sorted(os.listdir(adapter_save_path)):
    fpath = os.path.join(adapter_save_path, fname)
    size_mb = os.path.getsize(fpath) / 1024 / 1024
    print(f"  {fname}: {size_mb:.1f} MB")

# プロジェクトresultsにもメタデータを保存
results_metadata_file = f"{PROJECT_PATH}/results/phase9c_step3_training_metadata.json"
with open(results_metadata_file, "w", encoding="utf-8") as f:
    json.dump(training_metadata, f, ensure_ascii=False, indent=2)
print(f"\nTraining metadata also saved to: {results_metadata_file}")

✅ Adapter saved to: /content/drive/MyDrive/models/qwen3-32b-poi-qlora

Saved files:
  README.md: 0.0 MB
  adapter_config.json: 0.0 MB
  adapter_model.safetensors: 512.1 MB
  chat_template.jinja: 0.0 MB
  tokenizer.json: 10.9 MB
  tokenizer_config.json: 0.0 MB
  training_metadata.json: 0.0 MB

Training metadata also saved to: /content/drive/MyDrive/experiments-local-llm/results/phase9c_step3_training_metadata.json


## 7. Quick Test（推論テスト 3問）

In [ ]:
# FTモデルの推論テスト
test_questions = [
    "渋谷駅周辺のスターバックスはありますか？",
    "新宿駅から最も近いコンビニはどこですか？",
    "池袋駅と東京駅、カフェが多いのはどちらですか？",
]

model.eval()

for i, question in enumerate(test_questions):
    print(f"\n{'='*60}")
    print(f"Q{i+1}: {question}")
    print(f"{'='*60}")

    messages = [
        {"role": "system", "content": "あなたは東京都内の主要駅周辺エリアの地理情報に詳しいアシスタントです。結論、根拠、補足の構造で回答してください。"},
        {"role": "user", "content": question}
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.8,
            top_k=20,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    answer = re.sub(r'<think>.*?</think>', '', answer, flags=re.DOTALL).strip()
    answer = re.sub(r'</think>', '', answer).strip()

    print(f"A: {answer[:500]}")

    del inputs, outputs
    torch.cuda.empty_cache()

print(f"\n\u2705 Quick test complete")


Q1: 渋谷駅周辺のスターバックスはありますか？
A: 【結論】  
はい、渋谷駅周辺にはスターバックスがあります。

【根拠】  
スターバックスの公式サイトや地図サービス（Google Mapsなど）を参照すると、渋谷駅周辺には複数のスターバックス店舗が存在しています。例えば、渋谷駅の東口や南口周辺には商業施設や飲食店が集まり、スターバックスもその中に含まれています。具体的には、渋谷ヒカリエや渋谷マークシティなど、主要な商業施設内や近くにスターバックスが展開されています。

【補足】  
渋谷駅周辺には多くのカフェや飲食店が存在するため、スターバックス以外にも選択肢は豊富です。ただし、スターバックスは東口や南口周辺に集中しており、西口や北口方面には少ない傾向があります。利用予定の際は、最寄り駅出口やルートに応じて位置を確認することをおすすめします。

Q2: 新宿駅から最も近いコンビニはどこですか？
A: 【結論】  
新宿駅から最も近いコンビニは「ローソン」で、最短で約100mの距離に位置しています。

【根拠】  
新宿駅周辺のコンビニ情報によると、最も近いのは南東方向にあるローソン（100m）です。次に近いのは「ミニストップ」（112m）と「セブン-イレブン」（127m）となっています。これらの距離は、新宿駅の構内や周辺の主要出口からの最短距離を指しており、徒歩で数秒から数十秒で到着可能です。

【補足】  
新宿駅は複数の駅構内（JR、小田急、東急、西武等）と連絡が取れているため、現在地によって最寄りのコンビニが異なる可能性があります。ただし、最短距離としてはローソンが最も近いとされています。

Q3: 池袋駅と東京駅、カフェが多いのはどちらですか？


## 8. 結果サマリー・次ステップ

### 学習結果

| 項目 | 値 |
|------|----|
| Train loss (final) | **TBD** |
| Eval loss (best) | **TBD** |
| Eval loss (final) | **TBD** |
| 過学習判定 | **TBD** |

### 次ステップ

1. `phase9c_step3_evaluation.ipynb` で 3 システム比較評価を実行
   - System A: RAG (C2) — 既存結果
   - System B: FT-only — LoRA モデル単体
   - System C: FT+RAG — LoRA モデル + RAG
2. 130 テストケース × 2 システム (B, C) = 260 クエリ
3. 推定時間: 約 3.6 時間（A100）

## 9. 結果をGitにコミット & プッシュ

In [ ]:
# ステージング + コミット + プッシュ
!git -C {REPO_PATH} add notebooks/phase9c_step3_qlora_training.ipynb
!git -C {REPO_PATH} add results/phase9c_step3_training_metadata.json
!git -C {REPO_PATH} diff --cached --stat

!git -C {REPO_PATH} commit -m "results: Phase 9-C Step 3 QLoRA学習完了 (Qwen3-32B, 59件)"
!git -C {REPO_PATH} push origin {BRANCH}

print("\n\u2705 学習結果をリモートブランチにプッシュしました")